In [22]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [11]:
load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [8]:
doc_loader = PyPDFLoader('../book.pdf')

docs = doc_loader.load()

In [10]:
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=OpenAIEmbeddings(model='text-embedding-3-small')
)

In [31]:
base_retriever = vector_store.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k': 5}
)

mqr_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs = {'k': 5}),
    llm=llm
)

In [37]:
query = "How to design efficient and scalable systems?"

base_ret_results = base_retriever.invoke(query)
base_ret_results

[Document(id='f4cde3b6-c53a-429e-b854-f485cd466c0f', metadata={'total_pages': 269, 'creator': 'calibre 3.9.0 [https://calibre-ebook.com]', 'producer': 'calibre 3.9.0 [https://calibre-ebook.com]', 'creationdate': '2020-10-16T23:12:01+00:00', 'page': 31, 'source': '../book.pdf', 'page_label': '32', 'title': "System Design Interview – An insider's guide, Second Edition: Step by Step Guide, Tips and 15 System Design Interview Questions with Detailed Solutions", 'author': 'Alex Xu'}, page_content='Millions of users and beyond\nScaling a system is an iterative process. Iterating on what we have learned in this chapter\ncould get us far. More fine-tuning and new strategies are needed to scale beyond millions of\nusers. For example, you might need to optimize your system and decouple the system to even\nsmaller services. All the techniques learned in this chapter should provide a good foundation\nto tackle new challenges. To conclude this chapter, we provide a summary of how we scale\nour syst

In [38]:
mqr_results = mqr_retriever.invoke(query)
mqr_results

[Document(id='407229d8-360c-41a1-b9b1-dc7187c03c9e', metadata={'source': '../book.pdf', 'creationdate': '2020-10-16T23:12:01+00:00', 'page': 264, 'title': "System Design Interview – An insider's guide, Second Edition: Step by Step Guide, Tips and 15 System Design Interview Questions with Detailed Solutions", 'page_label': '265', 'total_pages': 269, 'creator': 'calibre 3.9.0 [https://calibre-ebook.com]', 'producer': 'calibre 3.9.0 [https://calibre-ebook.com]', 'author': 'Alex Xu'}, page_content='Real-world systems\nThe following materials can help you understand general design ideas of real system\narchitectures behind different companies.\nFacebook Timeline: Brought To You By The Power Of Denormalization:\nhttps://goo.gl/FCNrbm\nScale at Facebook: \nhttps://goo.gl/NGTdCs\nBuilding Timeline: Scaling up to hold your life story: \nhttps://goo.gl/8p5wDV\nErlang at Facebook (Facebook chat): \nhttps://goo.gl/zSLHrj\nFacebook Chat: \nhttps://goo.gl/qzSiWC\nFinding a needle in Haystack: Facebo

In [39]:
context_by_base_ret = "\n\n".join(doc.page_content for doc in base_ret_results)
context_by_mqr = "\n\n".join(doc.page_content for doc in mqr_results)

In [40]:
prompt = PromptTemplate(
    template = '''You are a helpful AI assistant.

    Answer the question using ONLY the provided context.
    If the answer is not available in the context, say so.

    Context:
    {context}

    Question:
    {question}''',

    input_variables=['context', 'question']
)

In [43]:
parser = StrOutputParser()

chain = prompt | llm | parser

base_ret_llm_results = chain.invoke({
    'context': context_by_base_ret,
    'question': query
})

mqr_ret_llm_results = chain.invoke({
    'context': context_by_mqr,
    'question': query
})

print(base_ret_llm_results)
print("\n", "MQR retriever: \n", mqr_ret_llm_results)

To design efficient and scalable systems, you should follow these techniques:

1. Keep the web tier stateless.
2. Build redundancy at every tier.
3. Cache data as much as you can.
4. Support multiple data centers.
5. Host static assets in a CDN.
6. Scale your data tier by sharding.
7. Split tiers into individual services.
8. Monitor your system and use automation tools.

These strategies provide a good foundation for tackling the challenges of scaling systems to support millions of users.

 MQR retriever: 
 To design efficient and scalable systems, consider the following strategies:

1. **Keep web tier stateless**: This allows for easier scaling and management.
2. **Build redundancy at every tier**: This helps prevent single points of failure.
3. **Cache data as much as you can**: Caching improves performance by reducing load on databases.
4. **Support multiple data centers**: This enhances availability and reduces latency for users in different regions.
5. **Host static assets in CDN*